# English-to-French Translation Chatbot

This notebook builds an interactive chatbot that translates English text into French using a pre-trained Hugging Face translation model (`Helsinki-NLP/opus-mt-en-fr`). The translated response is streamed to the screen word by word to simulate a natural, real-time chatbot reply.

**Contents:**
1. Install required libraries
2. Import libraries
3. Load the pre-trained translation model
4. Streaming output helper
5. Translation function
6. Chatbot loop

Run the cells in order, from top to bottom.

## 1. Install Required Libraries

`transformers` provides access to pre-trained NLP models. `sentencepiece` is the tokenizer required by the MarianMT (Helsinki-NLP) models. `torch` and `torchvision` provide the underlying deep learning backend.

In [ ]:
!pip install -q -U torch torchvision transformers sentencepiece

## 2. Import Libraries

`time` and `sys` support the word-by-word streaming effect used to display chatbot responses. `textwrap` is imported for optional text formatting.

In [ ]:
from transformers import pipeline
import time
import sys
import textwrap

## 3. Load the Pre-trained Translation Model

We use `Helsinki-NLP/opus-mt-en-fr`, a MarianMT model hosted on the Hugging Face Hub and trained specifically for English-to-French translation. `AutoTokenizer` converts raw text into the numerical format the model expects, and `AutoModelForSeq2SeqLM` performs the sequence-to-sequence translation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

## 4. Streaming Output Helper

To make the chatbot feel more like a live conversation, the translated text is printed one word at a time with a short delay between words, instead of appearing all at once.

In [ ]:
def stream(text):
  words = text.split()
  for word in words:
    print(word, end=' ', flush=True)
    time.sleep(0.1)

## 5. Translation Function

This function tokenizes an English sentence, passes it through the model to generate a French translation, and decodes the result back into readable text.

In [ ]:
def translate_to_french(text):
    """
    Translate a piece of English text into French using the Hugging Face
    translation model.

    Parameters:
        text (str): The English sentence to translate.

    Returns:
        str: The translated French sentence.
    """
    inputs = tokenizer(text, return_tensors="pt")
    translated_tokens = model.generate(**inputs)
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text


# Quick sanity check
sample = "What are you doing?"
print(f"English: {sample}")
print("French: ", end="")
stream(translate_to_french(sample))


## 6. Chatbot Loop

This is the main interactive loop. The user types an English sentence, the model translates it into French, and the result is streamed back to the screen. Typing `exit`, `quit`, or `bye` ends the conversation.

In [ ]:
def run_chatbot():
    """
    Main chatbot loop.
    Continuously accepts English input from the user, translates it to
    French, and displays the result with a typing animation until the
    user chooses to exit.
    """
    exit_words = {"exit", "quit", "bye"}


    print(" English to French Translator")
    print(" Type a sentence in English, or 'exit' to stop.")


    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            # Skip empty input instead of sending it to the model
            print("Chatbot: Please type something in English!")
            continue

        if user_input.lower() in exit_words:
            stream("Au revoir! (Goodbye!)")
            break

        # Translate the input text using our Hugging Face model
        french_translation = translate_to_french(user_input)

        # Display the translation with the typing effect
        print("French: ", end="")
        stream(french_translation)


run_chatbot()